In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report

In [21]:
data = pd.read_csv('data/FilteredZooSpecPhotoDR19.csv')
data['target'] = data['spiral'] # 1 is spiral, 0 is elliptical
svd_output = pd.read_csv('data/SVD_Pixels.csv')

final_data = data.merge(svd_output,how='inner',on='objid')

In [25]:
train_split = 0.6
val_split = 0.2
test_split = 0.2

X = final_data.copy()
X = X.drop(columns=['objid','specobjid','ra','dec','spiral','elliptical'])

train, temp = train_test_split(X, test_size=(1-train_split), stratify=X['target'],random_state=111)

rel_val_size = val_split / (val_split + test_split)
val, test = train_test_split(temp, test_size=(1-rel_val_size), stratify=temp['target'],random_state =111)

train = train.reset_index(drop=True)
val = val.reset_index(drop=True)
test = test.reset_index(drop=True)

In [27]:
X_train = train.drop(columns=['target'])
y_train = train['target']

X_val = val.drop(columns=['target'])
y_val = val['target']

X_test = test.drop(columns=['target'])
y_test = test['target']

y_train = y_train.astype(int)
y_val = y_val.astype(int)
y_test = y_test.astype(int)


In [29]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

In [31]:

rf = RandomForestClassifier(random_state=111)
svm = SVC(kernel='rbf', random_state=111)
logreg = LogisticRegression(max_iter=500, multi_class='multinomial', random_state=111)

# Hyperparameter grids
param_grids = {
    'Random Forest': {'n_estimators': [100, 200], 'max_depth': [None, 10, 20]},
    'SVM': {'C': [0.1, 1, 10], 'gamma': ['scale', 'auto']},
    'Logistic Regression': {'C': [0.1, 1, 10]}
}

In [33]:


best_models = {}

# Random Forest
best_acc = 0
for n in [100, 150, 200, 250]:
    for d in [1, 10, 15]:
        rf_tmp = RandomForestClassifier(n_estimators=n, max_depth=d, random_state=111)
        rf_tmp.fit(X_train, y_train)
        y_val_pred = rf_tmp.predict(X_val)
        acc = accuracy_score(y_val, y_val_pred)
        if acc > best_acc:
            best_acc = acc
            best_models['Random Forest'] = rf_tmp
print(f"Best RF val acc: {best_acc:.3f}")

# SVM
best_acc = 0
for C in [0.1, 1, 3, 5, 7, 10]:
    for gamma in ['scale', 'auto']:
        svm_tmp = SVC(C=C, gamma=gamma, kernel='rbf', random_state=111)
        svm_tmp.fit(X_train_scaled, y_train)
        y_val_pred = svm_tmp.predict(X_val_scaled)
        acc = accuracy_score(y_val, y_val_pred)
        if acc > best_acc:
            best_acc = acc
            best_models['SVM'] = svm_tmp
print(f"Best SVM val acc: {best_acc:.3f}")

# Logistic Regression
best_acc = 0
for C in [0.1, 1, 3, 5, 7, 10]:
    logreg_tmp = LogisticRegression(C=C, max_iter=500, multi_class='multinomial', random_state=111)
    logreg_tmp.fit(X_train_scaled, y_train)
    y_val_pred = logreg_tmp.predict(X_val_scaled)
    acc = accuracy_score(y_val, y_val_pred)
    if acc > best_acc:
        best_acc = acc
        best_models['Logistic Regression'] = logreg_tmp
print(f"Best Logistic Regression val acc: {best_acc:.3f}")


Best RF val acc: 0.985
Best SVM val acc: 0.983


C:\Users\marce\anaconda3\envs\pythonbootcamp\Lib\site-packages\sklearn\linear_model\_logistic.py:1262: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, binary problems will be fit as proper binary  logistic regression models (as if multi_class='ovr' were set). Leave it to its default value to avoid this warning.
  warnings.warn(
C:\Users\marce\anaconda3\envs\pythonbootcamp\Lib\site-packages\sklearn\linear_model\_logistic.py:1262: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, binary problems will be fit as proper binary  logistic regression models (as if multi_class='ovr' were set). Leave it to its default value to avoid this warning.
  warnings.warn(
C:\Users\marce\anaconda3\envs\pythonbootcamp\Lib\site-packages\sklearn\linear_model\_logistic.py:1262: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, binary problems will be fit as

Best Logistic Regression val acc: 0.982


C:\Users\marce\anaconda3\envs\pythonbootcamp\Lib\site-packages\sklearn\linear_model\_logistic.py:1262: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, binary problems will be fit as proper binary  logistic regression models (as if multi_class='ovr' were set). Leave it to its default value to avoid this warning.
  warnings.warn(
C:\Users\marce\anaconda3\envs\pythonbootcamp\Lib\site-packages\sklearn\linear_model\_logistic.py:1262: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, binary problems will be fit as proper binary  logistic regression models (as if multi_class='ovr' were set). Leave it to its default value to avoid this warning.
  warnings.warn(


In [35]:


for name, model in best_models.items():
    if name in ['SVM', 'Logistic Regression']:
        X_used = X_test_scaled
    else:
        X_used = X_test

    y_pred = model.predict(X_used)
    print(f"\n=== {name} Test Results ===")
    print(classification_report(y_test, y_pred))



=== Random Forest Test Results ===
              precision    recall  f1-score   support

           0       1.00      0.95      0.98       879
           1       0.99      1.00      0.99      2715

    accuracy                           0.99      3594
   macro avg       0.99      0.98      0.98      3594
weighted avg       0.99      0.99      0.99      3594


=== SVM Test Results ===
              precision    recall  f1-score   support

           0       1.00      0.95      0.97       879
           1       0.98      1.00      0.99      2715

    accuracy                           0.99      3594
   macro avg       0.99      0.97      0.98      3594
weighted avg       0.99      0.99      0.99      3594


=== Logistic Regression Test Results ===
              precision    recall  f1-score   support

           0       0.99      0.95      0.97       879
           1       0.99      1.00      0.99      2715

    accuracy                           0.99      3594
   macro avg       0.99 

# Interpretation of Results

All of the models considered were near perfect. This gives us the hint that the dataset is very easy to linearly separate.

In [44]:
weights = best_models['Logistic Regression'].coef_[0]

In [46]:
weights

array([-9.42005196e+00,  4.57948512e+00, -6.23452323e-01, -2.53959116e-01,
       -2.19640622e-01,  1.26272541e+00, -2.56684156e+00, -5.15720905e-01,
        8.01136423e-01,  3.94620504e-02,  3.94664403e-02,  3.94640375e-02,
        3.94642394e-02,  3.94672702e-02, -6.36045912e-02, -2.59659383e-03,
       -3.80554764e-02,  8.13862511e-02, -2.53448352e-01,  3.21151739e-01,
        9.81204487e-02,  9.91137242e-02, -2.47887640e-01,  1.33833878e-01])

In [52]:
abs_weights = np.abs(weights)

In [60]:
feature_names = X_train.columns

In [66]:
len(feature_names)

24

In [71]:
best_features = pd.Series(abs_weights, index=X_train.columns).sort_values(ascending=False)

In [73]:
best_features

p_el_debiased    9.420052
p_cs_debiased    4.579485
modelMag_r       2.566842
modelMag_g       1.262725
modelMag_z       0.801136
petroR50_r       0.623452
modelMag_i       0.515721
svd_comp_6       0.321152
petroR90_r       0.253959
svd_comp_5       0.253448
svd_comp_9       0.247888
modelMag_u       0.219641
svd_comp_10      0.133834
svd_comp_8       0.099114
svd_comp_7       0.098120
svd_comp_4       0.081386
svd_comp_1       0.063605
extinction_z     0.039467
extinction_g     0.039466
extinction_i     0.039464
extinction_r     0.039464
extinction_u     0.039462
svd_comp_3       0.038055
svd_comp_2       0.002597
dtype: float64